# 01 Explore: Karnataka daily wind and solar, gaps, flags, NASA weather join

Exploration only: no features and no models. Everything is read through the `solarwind` package from `data/processed` and `data/raw` (run `dvc pull` first on a fresh clone).

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from solarwind.config import get_settings, load_sites
from solarwind.data.cea.build_daily import missing_days
from solarwind.data.join import join_with_weather, source_weather
from solarwind.data.tables import load_daily, load_flags, load_parse_failures

# settings paths are relative to the repository root
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(root)

settings = get_settings()
sites = load_sites(settings.sites_csv)
daily = load_daily(settings)
print("repo root:", root)
print(
    "daily rows:",
    len(daily),
    "| first",
    daily["date"].min().date(),
    "| last",
    daily["date"].max().date(),
)
pd.DataFrame([s.model_dump() for s in sites])

## 1. Karnataka daily wind and solar over time

`wind_mu` and `solar_mu` are the CEA `Generation` sheet values (state control plus ISGS), in million units (MU), net. Lines break where days are missing.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
full_index = pd.date_range(daily["date"].min(), daily["date"].max())
series = daily.set_index("date").reindex(full_index)
axes[0].plot(series.index, series["wind_mu"], color="tab:blue", linewidth=0.9)
axes[0].set_ylabel("wind (MU/day)")
axes[1].plot(series.index, series["solar_mu"], color="tab:orange", linewidth=0.9)
axes[1].set_ylabel("solar (MU/day)")
axes[0].set_title("Karnataka daily generation (CEA Generation sheet)")
fig.tight_layout()
daily[["wind_mu", "solar_mu"]].describe()

## 2. Missing days by month, and the rejected files

A missing day is a date between the first and last report with no row in the table. That includes dates the CEA portal does not list and the files the parser rejected.

In [ ]:
gaps = missing_days(daily)
by_month = gaps.to_series().groupby(gaps.to_period("M")).size()
print("missing days:", len(gaps))
ax = by_month.plot(kind="bar", figsize=(10, 3.5), color="tab:red")
ax.set_ylabel("missing days")
ax.set_title("Missing days by month")
by_month.to_frame("missing_days")

In [ ]:
failures = load_parse_failures(settings)
print("rejected files:", len(failures))
failures

## 3. Flagged dates

Flags are logged by the build, never fixed. Each block shows the flagged date with the two days either side (a date missing from the table has no row).

In [ ]:
flags = load_flags(settings)
print("flags:", len(flags))
flags

In [ ]:
window = pd.Timedelta(days=2)
for flagged in flags["date"].drop_duplicates().sort_values():
    near = daily[(daily["date"] >= flagged - window) & (daily["date"] <= flagged + window)]
    print(f"--- around {flagged.date()}: " + "; ".join(flags.loc[flags["date"] == flagged, "flag"]))
    print(near.to_string(index=False))

## 4. Join with NASA POWER weather

The solar target is joined to the mean weather of the four solar sites and the wind target to the mean of the four wind sites, with the equal weights from `config/sites.csv`. A joined day needs a CEA row and weather for that date.

In [ ]:
solar_weather = source_weather(settings, sites, "solar")
wind_weather = source_weather(settings, sites, "wind")
joined = join_with_weather(daily, solar_weather, wind_weather)
cea_only = daily.loc[~daily["date"].isin(joined["date"]), "date"]
print("CEA range    :", daily["date"].min().date(), "to", daily["date"].max().date())
print(
    "NASA range   :", solar_weather["date"].min().date(), "to", solar_weather["date"].max().date()
)
print("rows         : CEA", len(daily), "| NASA", len(solar_weather), "| joined", len(joined))
print("CEA dates without weather:", len(cea_only), "->", cea_only.dt.date.astype(str).tolist())
print("weather NaN cells:", int(joined.filter(regex="^(solar|wind)_[A-Z]").isna().sum().sum()))
joined.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
pairs = [
    (
        axes[0],
        "solar_ALLSKY_SFC_SW_DWN",
        "solar_mu",
        "tab:orange",
        "irradiance (kWh/m2/day)",
        "solar (MU/day)",
    ),
    (axes[1], "wind_WS50M", "wind_mu", "tab:blue", "wind speed at 50 m (m/s)", "wind (MU/day)"),
]
for ax, x, y, color, xlabel, ylabel in pairs:
    ax.scatter(joined[x], joined[y], s=8, alpha=0.5, color=color)
    r = joined[x].corr(joined[y])
    ax.set(xlabel=xlabel, ylabel=ylabel, title=f"{y} vs {x} (r = {r:.2f})")
fig.tight_layout()

## 5. Distributions and outliers per series

**Outlier rule.** A value is an outlier when it lies outside the Tukey fences `Q1 - k * IQR` and `Q3 + k * IQR`, with `k = settings.outlier_iqr_multiplier` (1.5). The fences are computed per series over its whole available history (CEA 2024-06-30 to 2026-10-08, NASA 2020-01-01 to 2026-09-30). Outliers are only counted and displayed here: nothing is removed or clipped. Weather columns are the equal-weight means of the four sites of each source type (prefix `solar_` or `wind_`).

In [ ]:
from solarwind.data.quality import outlier_mask, outlier_summary

weather_all = (
    solar_weather.set_index("date")
    .add_prefix("solar_")
    .join(wind_weather.set_index("date").add_prefix("wind_"))
)
print("outlier multiplier k =", settings.outlier_iqr_multiplier)
target_summary = outlier_summary(daily[["wind_mu", "solar_mu"]], settings)
weather_summary = outlier_summary(weather_all, settings)
pd.concat([target_summary, weather_summary], ignore_index=True).round(3)

In [ ]:
series = {
    "wind_mu": daily.set_index("date")["wind_mu"],
    "solar_mu": daily.set_index("date")["solar_mu"],
    **{col: weather_all[col] for col in weather_all.columns},
}
fig, axes = plt.subplots(3, 4, figsize=(15, 8))
for ax, (name, values) in zip(axes.ravel(), series.items(), strict=False):
    mask = outlier_mask(values, settings)
    ax.hist(values.dropna(), bins=40, color="tab:gray")
    ax.set_title(f"{name} ({int(mask.sum())} outliers)", fontsize=9)
for ax in axes.ravel()[len(series) :]:
    ax.axis("off")
fig.tight_layout()

In [ ]:
extreme = {name: values[outlier_mask(values, settings)] for name, values in series.items()}
for name in ("wind_mu", "solar_mu"):
    print(
        name, "outlier dates (first 10):", [d.date().isoformat() for d in extreme[name].index[:10]]
    )
    print(
        "   range of outliers:",
        extreme[name].min() if len(extreme[name]) else None,
        extreme[name].max() if len(extreme[name]) else None,
    )

**What this means for modelling.** Both targets are well behaved: about 1% of days fall outside the fences (8 wind days, all high-wind July 2026 peaks of 97.5 to 107.1 MU; 7 solar days, low values of 11.9 to 18.9 MU, plausibly overcast days). These look like real extremes (monsoon wind peaks, cloudy days), not data errors, so keep them and do not clip; prefer robust error measures such as MAE alongside RMSE. Wind speeds are right-skewed (3 to 4% flagged) and wind power rises steeply with speed, so a non-linear or transformed wind feature is worth testing. The global Tukey rule is blind to level shifts by season: the suspect February 2026 solar days (22.46 and 21.69 MU, State-Wise solar 0.00) sit just above the low fence of 19.85 and are not flagged, so suspect days need a seasonal or residual-based check, not this rule alone.

## 6. Seasonality

Monthly means and day-of-year patterns for generation (CEA window only) and weather (NASA 2020-2026). The CEA window has uneven coverage per calendar month (the table below counts the days behind each monthly mean), so monthly generation means are indicative.

In [ ]:
from solarwind.data.quality import day_of_year_profile, monthly_profile

gen_month = monthly_profile(daily[["date", "wind_mu", "solar_mu"]])
gen_month["days"] = daily.groupby(daily["date"].dt.month).size()
weather_month = monthly_profile(weather_all.reset_index())
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
axes[0, 0].bar(gen_month.index, gen_month["wind_mu"], color="tab:blue")
axes[0, 0].set_title("wind_mu by month (MU/day)")
axes[0, 1].bar(gen_month.index, gen_month["solar_mu"], color="tab:orange")
axes[0, 1].set_title("solar_mu by month (MU/day)")
axes[0, 2].bar(gen_month.index, gen_month["days"], color="tab:gray")
axes[0, 2].set_title("CEA days behind each monthly mean")
axes[1, 0].plot(weather_month.index, weather_month["solar_ALLSKY_SFC_SW_DWN"], marker="o")
axes[1, 0].set_title("solar sites: irradiance by month")
axes[1, 1].plot(weather_month.index, weather_month["wind_WS50M"], marker="o", color="tab:blue")
axes[1, 1].set_title("wind sites: WS50M by month")
axes[1, 2].plot(weather_month.index, weather_month["solar_T2M"], marker="o", color="tab:red")
axes[1, 2].set_title("solar sites: T2M by month")
fig.tight_layout()
gen_month.round(2)

In [ ]:
gen_doy = (
    day_of_year_profile(daily[["date", "wind_mu", "solar_mu"]])
    .rolling(15, center=True, min_periods=5)
    .mean()
)
weather_doy = (
    day_of_year_profile(weather_all.reset_index()).rolling(15, center=True, min_periods=5).mean()
)
fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True)
axes[0, 0].plot(gen_doy.index, gen_doy["wind_mu"], color="tab:blue")
axes[0, 0].set_title("wind_mu by day of year (15-day mean)")
axes[0, 1].plot(gen_doy.index, gen_doy["solar_mu"], color="tab:orange")
axes[0, 1].set_title("solar_mu by day of year (15-day mean)")
axes[1, 0].plot(weather_doy.index, weather_doy["wind_WS50M"], color="tab:blue")
axes[1, 0].set_title("wind sites WS50M by day of year (15-day mean)")
axes[1, 1].plot(weather_doy.index, weather_doy["solar_ALLSKY_SFC_SW_DWN"], color="tab:orange")
axes[1, 1].set_title("solar sites irradiance by day of year (15-day mean)")
for ax in axes[1]:
    ax.set_xlabel("day of year")
fig.tight_layout()

**What this means for modelling.** The two sources have opposite seasonality. Wind is high from June to September (monthly means 54 to 75 MU/day in June to August) and low from October to May (17 to 29 MU/day). Solar is highest from January to May (48 to 53 MU/day) and lowest in July and August (37.5 to 41.7 MU/day), when the monsoon cuts irradiance. Each monthly mean rests on only 50 to 88 days (at most two years, with the May to August 2025 gaps), so the seasonal profile is noisy. The time-based split puts training on 2024-06-30 to 2025-12-31 and testing on January to September 2026, so the test window has no October to December: results must be read per season, and the model cannot be judged on the post-monsoon months. Calendar features (cyclical month and day of year) will carry the seasonal level; weather should explain day-to-day variation. Capacity also grows over the period (see the blocked section), which is confounded with season until a capacity decision is made.

## 7. Weather vs generation correlations, per source

Same-day Pearson and Spearman correlations between each source's weather variables (mean of its four sites) and its generation, over the joined days (CEA rows with NASA weather). Correlation is descriptive here; no feature is built from it yet.

In [ ]:
rows = []
for source, target in (("solar", "solar_mu"), ("wind", "wind_mu")):
    for variable in settings.nasa_parameters:
        column = f"{source}_{variable}"
        rows.append(
            {
                "source": source,
                "weather variable": variable,
                "pearson": joined[column].corr(joined[target]),
                "spearman": joined[column].corr(joined[target], method="spearman"),
            }
        )
corr = pd.DataFrame(rows)
print("joined days:", len(joined))
corr.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, source in zip(axes, ("solar", "wind"), strict=True):
    part = corr[corr["source"] == source].set_index("weather variable")
    part[["pearson", "spearman"]].plot(kind="bar", ax=ax, rot=0)
    ax.axhline(0, color="black", linewidth=0.6)
    ax.set_title(f"{source}_mu vs {source} site weather")
fig.tight_layout()

**What this means for modelling.** Each source has one dominant weather driver: wind_mu follows wind speed (r = 0.92 with WS10M, 0.89 with WS50M) and solar_mu follows irradiance (r = 0.81). Secondary links are weak or seasonal: temperature is mildly positive for solar (0.29), while wind speed is negatively related to solar output (-0.35) and irradiance to wind output (-0.39), because windy days in the monsoon are also cloudy; these are not causal and should not be fed to the wrong source model blindly. Spearman values are close to Pearson, so a linear baseline on the dominant driver should already do well, and the gap left for non-linear models is probably in wind (power rises with speed) and in the cloud and season effects. Use source-specific weather features plus calendar terms, and judge each model per season.

## 8. Capacity growth (blocked)

**Blocked on the capacity decision.** Karnataka solar and wind capacity over time cannot be plotted until a capacity source and solar definition are chosen (HANDOFF section 5.5). Placeholder: capacity growth chart and its alignment with the daily generation series.

## 9. Capacity-factor sanity check (blocked)

**Blocked on the capacity decision.** The check CF = mean daily MU x 1000 / (MW x 24) against the expected solar range 0.15 to 0.25 needs monthly capacity. Placeholder: monthly CF per source with out-of-range months flagged.